# Showcase analysis — AS+OFI on real Binance L1 data

Loads the 12 PnL CSVs produced by `scripts/run_showcase.sh` (3 symbols × 4 strategies),
computes per-cell metrics, and renders the two showcase figures the README/`docs/showcase.md` embed.

**Inputs:** `data/showcase/{BTCUSDT,ETHUSDT,SOLUSDT}/pnl_{baseline,as,as_ofi,full}.csv`  
**Outputs:** `data/showcase/pnl_curves.png`, `data/showcase/sharpe_fillrate.png`, `data/showcase/results_table.md`

In [1]:
import csv, glob, math, os, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(ROOT) != 'electronic_trading_ecosystem':
    ROOT = os.getcwd()
SHOWCASE_DIR = os.path.join(ROOT, 'data', 'showcase')
SYMBOLS    = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
STRATEGIES = ['baseline', 'as', 'as_ofi', 'full']
MAKER_FEE  = float(os.environ.get('MAKER_FEE', '0.0002'))
print(f'ROOT={ROOT}\nSHOWCASE_DIR={SHOWCASE_DIR}\nMAKER_FEE={MAKER_FEE*1e4:.1f}bps')

ROOT=/Users/arkaj/Desktop/projects/trading-system-project/electronic_trading_ecosystem
SHOWCASE_DIR=/Users/arkaj/Desktop/projects/trading-system-project/electronic_trading_ecosystem/data/showcase
MAKER_FEE=2.0bps


In [2]:
def _stats(xs):
    if not xs:
        return 0.0, 0.0
    m = sum(xs) / len(xs)
    v = sum((x - m) ** 2 for x in xs) / len(xs)
    return m, math.sqrt(v)

def load_pnl(path):
    rows = []
    with open(path) as f:
        r = csv.DictReader(f)
        for row in r:
            rows.append({
                'ts_ns':        int(row['ts_ns']),
                'mid':          float(row['mid']),
                'position':     int(row['position']),
                'total_pnl':    float(row['total_pnl']),
                'volume':       int(row['volume']),
                'num_fills':    int(row['num_fills']),
                'num_requotes': int(row['num_requotes']),
            })
    return rows

def metrics(rows):
    if not rows:
        return {}
    last = rows[-1]
    pnl_series = [r['total_pnl'] for r in rows]
    pnl_diffs  = [b - a for a, b in zip(pnl_series, pnl_series[1:])]
    pos_series = [r['position']  for r in rows]
    avg_mid    = sum(r['mid'] for r in rows) / len(rows)
    notional   = last['volume'] * avg_mid
    fees       = notional * MAKER_FEE
    mean_ret, sd_ret = _stats(pnl_diffs)
    sharpe_step = (mean_ret / sd_ret) if sd_ret > 0 else 0.0
    sharpe_tape = sharpe_step * math.sqrt(max(len(pnl_diffs), 1))
    peak, max_dd = pnl_series[0], 0.0
    for p in pnl_series:
        peak = max(peak, p)
        max_dd = min(max_dd, p - peak)
    return {
        'gross_pnl': last['total_pnl'],
        'fees':      fees,
        'fee_adj':   last['total_pnl'] - fees,
        'sharpe':    sharpe_tape,
        'max_dd':    max_dd,
        'fills':     last['num_fills'],
        'requotes':  last['num_requotes'],
        'fill_rate': last['num_fills'] / max(last['num_requotes'], 1),
        'pos_std':   _stats(pos_series)[1],
        'rows':      len(rows),
    }

data = {}
for sym in SYMBOLS:
    data[sym] = {}
    for strat in STRATEGIES:
        path = os.path.join(SHOWCASE_DIR, sym, f'pnl_{strat}.csv')
        if not os.path.exists(path):
            print(f'MISSING: {path}', file=sys.stderr)
            continue
        rows = load_pnl(path)
        m    = metrics(rows)
        m['rows_data'] = rows  # keep for plotting
        data[sym][strat] = m
        print(f'  {sym:8s} {strat:9s}  rows={len(rows):>6d}  gross={m["gross_pnl"]:>10.1f}  fee_adj={m["fee_adj"]:>10.1f}  sharpe={m["sharpe"]:>5.2f}  fills={m["fills"]}')

  BTCUSDT  baseline   rows= 99221  gross=-2160327.0  fee_adj=-13707684.5  sharpe=-4.59  fills=75107


  BTCUSDT  as         rows= 99221  gross=-2440200.0  fee_adj=-11175861.0  sharpe=-28.08  fills=57691


  BTCUSDT  as_ofi     rows= 99221  gross=-2332768.0  fee_adj=-10875383.4  sharpe=-27.00  fills=56492


  BTCUSDT  full       rows= 99221  gross=-1784625.0  fee_adj=-8107912.6  sharpe=-32.53  fills=42007


  ETHUSDT  baseline   rows=125210  gross=-3613376.0  fee_adj=-13206282.4  sharpe=-10.33  fills=94310


...

  ETHUSDT  full       rows=125210  gross=-2573151.0  fee_adj=-9246159.9  sharpe=-25.34  fills=69392


  SOLUSDT  baseline   rows=132047  gross=-2535938.0  fee_adj=-9729507.6  sharpe=-7.97  fills=72585


  SOLUSDT  as         rows=132047  gross=-2612641.0  fee_adj=-10549655.6  sharpe=-40.52  fills=77238


  SOLUSDT  as_ofi     rows=132047  gross=-1303927.0  fee_adj=-7015439.0  sharpe=-27.74  fills=57788


  SOLUSDT  full       rows=132047  gross=-1658036.0  fee_adj=-7084402.2  sharpe=-24.18  fills=61275


In [3]:
# Figure 1: PnL curves faceted by symbol (3 panels × 4 strategies = 12 lines).
COLORS = {'baseline':'#888888', 'as':'#4c72b0', 'as_ofi':'#dd8452', 'full':'#55a868'}
fig, axes = plt.subplots(1, len(SYMBOLS), figsize=(15, 5), sharey=False)
if len(SYMBOLS) == 1:
    axes = [axes]
for ax, sym in zip(axes, SYMBOLS):
    for strat in STRATEGIES:
        cell = data.get(sym, {}).get(strat)
        if not cell:
            continue
        rows = cell['rows_data']
        ts   = [r['ts_ns']/1e9 for r in rows]
        pnl  = [r['total_pnl'] for r in rows]
        ax.plot(ts, pnl, label=strat, color=COLORS.get(strat, None), linewidth=1.4)
    ax.set_title(f'{sym} — 2024-03-28 (Binance USD-M perp)')
    ax.set_xlabel('simulated time [s]')
    ax.set_ylabel('total PnL (quote units)')
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=9)
    ax.axhline(0, color='k', linewidth=0.5, alpha=0.4)
fig.suptitle('AS+OFI maker — gross PnL across BTC / ETH / SOL (2024-03-28)', fontsize=13, y=1.03)
fig.tight_layout()
out1 = os.path.join(SHOWCASE_DIR, 'pnl_curves.png')
fig.savefig(out1, dpi=140, bbox_inches='tight')
print(f'wrote {out1}')

wrote /Users/arkaj/Desktop/projects/trading-system-project/electronic_trading_ecosystem/data/showcase/pnl_curves.png


In [4]:
# Figure 2: Sharpe + fill rate, averaged across symbols, grouped by strategy.
def mean_metric(strat, key):
    vals = [data[s][strat][key] for s in SYMBOLS if strat in data.get(s, {})]
    return sum(vals) / len(vals) if vals else 0.0

sharpe_by_strat    = [mean_metric(s, 'sharpe')    for s in STRATEGIES]
fillrate_by_strat  = [mean_metric(s, 'fill_rate') for s in STRATEGIES]
fee_adj_by_strat   = [mean_metric(s, 'fee_adj')   for s in STRATEGIES]

fig, (ax_s, ax_f, ax_p) = plt.subplots(1, 3, figsize=(15, 4.5))
x = list(range(len(STRATEGIES)))
bar_colors = [COLORS[s] for s in STRATEGIES]
ax_s.bar(x, sharpe_by_strat,   color=bar_colors)
ax_s.set_xticks(x); ax_s.set_xticklabels(STRATEGIES); ax_s.set_ylabel('Sharpe (per-tape)')
ax_s.set_title('Sharpe — mean across symbols'); ax_s.grid(alpha=0.3, axis='y')
ax_s.axhline(0, color='k', linewidth=0.5, alpha=0.4)
ax_f.bar(x, [v*100 for v in fillrate_by_strat], color=bar_colors)
ax_f.set_xticks(x); ax_f.set_xticklabels(STRATEGIES); ax_f.set_ylabel('fill rate [%]')
ax_f.set_title('Fill rate — fills / requotes'); ax_f.grid(alpha=0.3, axis='y')
ax_p.bar(x, fee_adj_by_strat,  color=bar_colors)
ax_p.set_xticks(x); ax_p.set_xticklabels(STRATEGIES); ax_p.set_ylabel('fee-adj PnL (quote)')
ax_p.set_title('Fee-adjusted PnL — mean across symbols (maker=2bps)')
ax_p.grid(alpha=0.3, axis='y'); ax_p.axhline(0, color='k', linewidth=0.5, alpha=0.4)
fig.tight_layout()
out2 = os.path.join(SHOWCASE_DIR, 'sharpe_fillrate.png')
fig.savefig(out2, dpi=140, bbox_inches='tight')
print(f'wrote {out2}')

wrote /Users/arkaj/Desktop/projects/trading-system-project/electronic_trading_ecosystem/data/showcase/sharpe_fillrate.png


In [5]:
# Results table (Markdown) — emitted to data/showcase/results_table.md for
# the showcase doc to include.
lines = []
lines.append('| Symbol | Strategy | Gross PnL | Fees (2bp) | Fee-adj | Sharpe | Max DD | Fills | Fill rate | Inv σ |')
lines.append('|--------|----------|-----------|-----------|---------|--------|--------|-------|-----------|-------|')
for sym in SYMBOLS:
    for strat in STRATEGIES:
        m = data.get(sym, {}).get(strat)
        if not m:
            continue
        lines.append(f"| {sym} | {strat} | {m['gross_pnl']:.1f} | {m['fees']:.1f} | {m['fee_adj']:.1f} | {m['sharpe']:.2f} | {m['max_dd']:.1f} | {m['fills']} | {m['fill_rate']*100:.1f}% | {m['pos_std']:.1f} |")

# Headline: best strategy by fee-adjusted PnL across all 12 cells.
winners = []
for sym in SYMBOLS:
    cells = [(s, data[sym][s]['fee_adj']) for s in STRATEGIES if s in data.get(sym, {})]
    if cells:
        winners.append((sym, max(cells, key=lambda c: c[1])))

header = '\n## Per-symbol winners (fee-adjusted PnL)\n\n'
for sym, (strat, val) in winners:
    header += f'- **{sym}:** `{strat}` (fee_adj={val:.1f})\n'

out_md = os.path.join(SHOWCASE_DIR, 'results_table.md')
with open(out_md, 'w') as f:
    f.write('\n'.join(lines))
    f.write(header)
print(f'wrote {out_md}')
print('\n'.join(lines[:6] + ['...']))

wrote /Users/arkaj/Desktop/projects/trading-system-project/electronic_trading_ecosystem/data/showcase/results_table.md
| Symbol | Strategy | Gross PnL | Fees (2bp) | Fee-adj | Sharpe | Max DD | Fills | Fill rate | Inv σ |
|--------|----------|-----------|-----------|---------|--------|--------|-------|-----------|-------|
| BTCUSDT | baseline | -2160327.0 | 11547357.5 | -13707684.5 | -4.59 | -2252830.0 | 75107 | 102.5% | 114.6 |
| BTCUSDT | as | -2440200.0 | 8735661.0 | -11175861.0 | -28.08 | -2440760.5 | 57691 | 51.5% | 16.8 |
| BTCUSDT | as_ofi | -2332768.0 | 8542615.4 | -10875383.4 | -27.00 | -2333495.0 | 56492 | 49.0% | 16.2 |
| BTCUSDT | full | -1784625.0 | 6323287.6 | -8107912.6 | -32.53 | -1784670.0 | 42007 | 51.8% | 14.5 |
...
